In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 08:47:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 08:48:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/04 08:48:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/04 08:48:01 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/04 08:48:01 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/03/04 08:48:01 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [2]:
spark.version

'3.5.0'

In [78]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [79]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [80]:
import pandas as pd

df_zones = spark.read \
    .option("header", "true") \
    .csv('zones/taxi_zone_lookup.csv')

In [81]:
df_zones.toPandas()['Borough'].unique()

array(['EWR', 'Queens', 'Bronx', 'Manhattan', 'Staten Island', 'Brooklyn',
       'Unknown', 'N/A'], dtype=object)

In [82]:
df_zones.toPandas()['service_zone'].unique()

array(['EWR', 'Boro Zone', 'Yellow Zone', 'Airports', 'N/A'], dtype=object)

In [48]:
df.registerTempTable('yellowtable')


In [49]:
df_zones.registerTempTable('zonetable')

## Question 1

Question 1. Install Spark and PySpark  
3.5.0

In [6]:
df = df.repartition(4)

In [7]:
df.rdd.getNumPartitions()

[Stage 1:====================================>                      (5 + 3) / 8]

4

In [10]:
df.write.parquet('pq/', mode='overwrite')

## Question 2
Read the November 2025 Yellow into a Spark Dataframe.  

Repartition the Dataframe to 4 partitions and save it to parquet.  

In [11]:
ls -lh pq/

total 98M
-rw-r--r-- 1 wali wali   0 Mar  4 08:49 _SUCCESS
-rw-r--r-- 1 wali wali 25M Mar  4 08:49 part-00000-15619231-b2c5-45b9-88ca-0edc3a46e179-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  4 08:49 part-00001-15619231-b2c5-45b9-88ca-0edc3a46e179-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  4 08:49 part-00002-15619231-b2c5-45b9-88ca-0edc3a46e179-c000.snappy.parquet
-rw-r--r-- 1 wali wali 25M Mar  4 08:49 part-00003-15619231-b2c5-45b9-88ca-0edc3a46e179-c000.snappy.parquet


## What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? 
Select the answer which most closely matches.  
- 6MB  
- **25MB**   
- 75MB   
- 100MB  

## Question 3

In [12]:
df_result = spark.sql("""
SELECT 
    count(1) as total_trips
FROM
    yellowtable
where DATE(tpep_pickup_datetime) = '2025-11-15';

    
""")

In [13]:
df_result.show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+




How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610  
- 102,340  
- **162,604**  
- 225,768  

## Question 4  

In [32]:
df_result = spark.sql("""
SELECT 
    (UNIX_TIMESTAMP(tpep_dropoff_datetime)- 
    UNIX_TIMESTAMP(tpep_pickup_datetime)) 
    /3600 
    as trip_duration_hours
FROM
    yellowtable

order by 
    trip_duration_hours desc

limit 1
    
""")
df_result.show()

+-------------------+
|trip_duration_hours|
+-------------------+
|  90.64666666666666|
+-------------------+




What is the length of the longest trip in the dataset in hours?  

- 22.7  
- 58.2  
- **90.6**    
- 134.5    

## Question 5


Spark's User Interface which shows the application's dashboard runs on which local port?

- 80  
- 443  
- **4040**    
- 8080  

## Question 6: Least frequent pickup location zone

In [86]:
df_result = spark.sql("""
SELECT 

      z.service_zone,
     z.Borough,
   
  
   count(1) as Frequency
 
FROM
    yellowtable as yt

join zonetable as z

on yt.PULocationID = z.LocationID
group by
1,2

order by Frequency
limit 10
    
""")
df_result.show()

[Stage 107:>                                                        (0 + 4) / 4]

+------------+-------------+---------+
|service_zone|      Borough|Frequency|
+------------+-------------+---------+
|   Boro Zone|Staten Island|      392|
|         EWR|          EWR|      593|
|         N/A|          N/A|     1897|
|         N/A|      Unknown|     6138|
|   Boro Zone|        Bronx|    34892|
|   Boro Zone|       Queens|   100942|
|   Boro Zone|    Manhattan|   142569|
|   Boro Zone|     Brooklyn|   154936|
|    Airports|       Queens|   273687|
| Yellow Zone|    Manhattan|  3465398|
+------------+-------------+---------+




Load the zone lookup data into a temp view in Spark:  

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv  

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?  

**Governor's Island/Ellis Island/Liberty Island**  
Arden Heights  
Rikers Island  
Jamaica Bay  